# 7. Vizualizacija i analiza podataka

Ovaj notebook provodi analizu podataka iz dimenzijskog modela (star sheme) pomocu SQL upita i Python vizualizacija.

**KPI-evi:**
1. Broj ticketa po projektu
2. Prosjecno vrijeme rjesavanja po prioritetu
3. Mjesecni trend volumena ticketa
4. Raspodjela workflow vremena (open / in_progress / waiting)
5. Top 10 tehnicara po opterecenju
6. Analiza rezolucija po projektu

**Preduvjeti:** Pokrenuti notebookove 5 (DDL) i 6 (ETL).

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv()

DB_USER = os.getenv('DB_USER', 'root')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_NAME = os.getenv('DB_NAME', 'fipu_srp_projekt')

if not DB_PASSWORD:
    raise ValueError("DB_PASSWORD nije postavljen!")

engine = create_engine(f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}/{DB_NAME}")
print(f"Spojeno na bazu: {DB_NAME}")

# Stil grafova
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

## 7.1 Broj ticketa po projektu

In [ ]:
df1 = pd.read_sql("""
    SELECT p.naziv_projekta AS projekt, COUNT(*) AS broj_ticketa
    FROM fact_support_tickets f
    JOIN dim_projekt p ON f.projekt_key = p.projekt_key
    GROUP BY p.naziv_projekta
    ORDER BY broj_ticketa DESC
""", engine)

fig, ax = plt.subplots()
bars = ax.barh(df1['projekt'], df1['broj_ticketa'], color='#378ADD')
ax.set_xlabel('Broj ticketa')
ax.set_title('Broj ticketa po projektu')
ax.invert_yaxis()
for bar, val in zip(bars, df1['broj_ticketa']):
    ax.text(val + 30, bar.get_y() + bar.get_height()/2, str(val), va='center', fontsize=9)
plt.tight_layout()
plt.show()

## 7.2 Prosjecno vrijeme rjesavanja po prioritetu

In [ ]:
df2 = pd.read_sql("""
    SELECT ps.razina_prioriteta AS prioritet,
           COUNT(*) AS broj_ticketa,
           ROUND(AVG(f.vrijeme_rjesavanja_sati), 1) AS avg_sati,
           ROUND(AVG(f.vrijeme_rjesavanja_sati) / 24, 1) AS avg_dana
    FROM fact_support_tickets f
    JOIN dim_prioritet_status ps ON f.prioritet_status_key = ps.prioritet_status_key
    WHERE f.vrijeme_rjesavanja_sati IS NOT NULL
      AND ps.razina_prioriteta != 'unknown'
    GROUP BY ps.razina_prioriteta
    ORDER BY avg_sati DESC
""", engine)

fig, ax = plt.subplots()
colors = {'Blocker':'#E24B4A', 'Highest':'#D85A30', 'High':'#EF9F27',
          'Medium':'#378ADD', 'Low':'#1D9E75', 'Lowest':'#5DCAA5'}
bar_colors = [colors.get(p, '#888780') for p in df2['prioritet']]
bars = ax.bar(df2['prioritet'], df2['avg_dana'], color=bar_colors)
ax.set_ylabel('Prosjecno vrijeme rjesavanja (dana)')
ax.set_title('Prosjecno vrijeme rjesavanja po prioritetu')
for bar, val in zip(bars, df2['avg_dana']):
    ax.text(bar.get_x() + bar.get_width()/2, val + 1, f'{val:.0f}d', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print(df2.to_string(index=False))

## 7.3 Mjesecni trend volumena ticketa

In [ ]:
df3 = pd.read_sql("""
    SELECT v.godina, v.mjesec, COUNT(*) AS broj_ticketa
    FROM fact_support_tickets f
    JOIN dim_vrijeme v ON f.vrijeme_key = v.vrijeme_key
    GROUP BY v.godina, v.mjesec
    ORDER BY v.godina, v.mjesec
""", engine)

df3['datum'] = pd.to_datetime(df3[['godina', 'mjesec']].assign(day=1))

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df3['datum'], df3['broj_ticketa'], color='#378ADD', linewidth=1.5)
ax.fill_between(df3['datum'], df3['broj_ticketa'], alpha=0.15, color='#378ADD')
ax.set_ylabel('Broj ticketa')
ax.set_title('Mjesecni trend kreiranih ticketa')
ax.xaxis.set_major_locator(mticker.MaxNLocator(12))
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 7.4 Raspodjela workflow vremena

In [ ]:
df4 = pd.read_sql("""
    SELECT
        ROUND(AVG(f.sati_open), 1) AS avg_open,
        ROUND(AVG(f.sati_in_progress), 1) AS avg_in_progress,
        ROUND(AVG(f.sati_resolved), 1) AS avg_resolved,
        ROUND(AVG(f.sati_waiting), 1) AS avg_waiting
    FROM fact_support_tickets f
    WHERE f.sati_open IS NOT NULL
""", engine)

labels = ['Open', 'In progress', 'Waiting', 'Resolved']
values = [df4['avg_open'].iloc[0], df4['avg_in_progress'].iloc[0],
          df4['avg_waiting'].iloc[0], df4['avg_resolved'].iloc[0]]
colors = ['#85B7EB', '#EF9F27', '#E24B4A', '#5DCAA5']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
ax1.pie(values, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax1.set_title('Udio prosjecnog vremena po stanju')

# Bar chart (sati)
bars = ax2.bar(labels, values, color=colors)
ax2.set_ylabel('Prosjecno sati')
ax2.set_title('Prosjecno vrijeme u svakom stanju (sati)')
for bar, val in zip(bars, values):
    ax2.text(bar.get_x() + bar.get_width()/2, val + 5, f'{val:.0f}h', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

## 7.5 Top 10 tehnicara po opterecenju

In [ ]:
df5 = pd.read_sql("""
    SELECT t.ime_prezime AS tehnicar,
           COUNT(*) AS broj_ticketa,
           ROUND(AVG(f.vrijeme_rjesavanja_sati) / 24, 1) AS avg_dana
    FROM fact_support_tickets f
    JOIN dim_tehnicar t ON f.assignee_key = t.tehnicar_key
    GROUP BY t.ime_prezime
    ORDER BY broj_ticketa DESC
    LIMIT 10
""", engine)

fig, ax1 = plt.subplots(figsize=(12, 6))
x = range(len(df5))
bars = ax1.bar(x, df5['broj_ticketa'], color='#378ADD', label='Broj ticketa')
ax1.set_ylabel('Broj ticketa', color='#378ADD')
ax1.set_title('Top 10 tehnicara — opterecenje i prosjecno vrijeme')
ax1.set_xticks(x)
ax1.set_xticklabels(df5['tehnicar'], rotation=45, ha='right')

ax2 = ax1.twinx()
ax2.plot(x, df5['avg_dana'], color='#E24B4A', marker='o', linewidth=2, label='Avg dana')
ax2.set_ylabel('Prosjecno vrijeme (dana)', color='#E24B4A')

fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
plt.tight_layout()
plt.show()

## 7.6 Analiza rezolucija po projektu

In [ ]:
df6 = pd.read_sql("""
    SELECT p.naziv_projekta AS projekt, ps.naziv_statusa AS status,
           COUNT(*) AS cnt
    FROM fact_support_tickets f
    JOIN dim_projekt p ON f.projekt_key = p.projekt_key
    JOIN dim_prioritet_status ps ON f.prioritet_status_key = ps.prioritet_status_key
    GROUP BY p.naziv_projekta, ps.naziv_statusa
    ORDER BY p.naziv_projekta, cnt DESC
""", engine)

# Pivot za stacked bar
pivot = df6.pivot_table(index='projekt', columns='status', values='cnt', fill_value=0)

# Odaberi top statuse (closed, done, ostali grupirani)
top_statuses = ['closed', 'done']
other_cols = [c for c in pivot.columns if c not in top_statuses]
pivot['ostalo'] = pivot[other_cols].sum(axis=1)
plot_df = pivot[top_statuses + ['ostalo']].sort_values('closed', ascending=True)

fig, ax = plt.subplots(figsize=(12, 6))
plot_df.plot(kind='barh', stacked=True, ax=ax, color=['#5DCAA5', '#378ADD', '#D3D1C7'])
ax.set_xlabel('Broj ticketa')
ax.set_title('Status ticketa po projektu')
ax.legend(title='Status')
plt.tight_layout()
plt.show()

## 7.7 Ticketi po danu u tjednu i kvartalu (heatmap)

In [ ]:
df7 = pd.read_sql("""
    SELECT v.dan_u_tjednu, v.kvartal, COUNT(*) AS cnt
    FROM fact_support_tickets f
    JOIN dim_vrijeme v ON f.vrijeme_key = v.vrijeme_key
    GROUP BY v.dan_u_tjednu, v.kvartal
""", engine)

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
pivot = df7.pivot_table(index='dan_u_tjednu', columns='kvartal', values='cnt', fill_value=0)
pivot = pivot.reindex(day_order)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, cmap='Blues', aspect='auto')
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'Q{q}' for q in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_title('Ticketi po danu u tjednu i kvartalu')

for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        val = pivot.values[i, j]
        color = 'white' if val > pivot.values.max() * 0.6 else 'black'
        ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=9, color=color)

plt.colorbar(im, ax=ax, label='Broj ticketa')
plt.tight_layout()
plt.show()

## 7.8 Sazetak KPI-eva

In [ ]:
kpi = pd.read_sql("""
    SELECT
        COUNT(*) AS ukupno_ticketa,
        COUNT(DISTINCT f.projekt_key) AS broj_projekata,
        COUNT(DISTINCT f.reporter_key) AS broj_reportera,
        COUNT(DISTINCT f.assignee_key) AS broj_assigneea,
        ROUND(AVG(f.vrijeme_rjesavanja_sati) / 24, 1) AS avg_dana_rjesavanja,
        ROUND(AVG(f.broj_komentara), 1) AS avg_komentara
    FROM fact_support_tickets f
""", engine)

print("=" * 50)
print("SAZETAK KPI-EVA")
print("=" * 50)
print(f"  Ukupno ticketa:            {kpi['ukupno_ticketa'].iloc[0]:,}")
print(f"  Broj projekata:            {kpi['broj_projekata'].iloc[0]}")
print(f"  Broj reportera:            {kpi['broj_reportera'].iloc[0]}")
print(f"  Broj assigneea:            {kpi['broj_assigneea'].iloc[0]}")
print(f"  Avg vrijeme rjesavanja:    {kpi['avg_dana_rjesavanja'].iloc[0]} dana")
print(f"  Avg komentara po ticketu:  {kpi['avg_komentara'].iloc[0]}")